# LoRA fine-tuning

Before starting to run cells top to bottom change runtime type to T4 GPU: Runtime > Change runtime type > pick a **T4 GPU** > Save.

If it says "too many active sessions," terminate background runtimes.

## Step 1: Check that you have a GPU

In [ ]:
!nvidia-smi

Tue Oct 20 22:31:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Step 2: Install the libraries (takes about 1 minute)

In [ ]:
!pip install -q -U peft accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 73.4 MB/s eta 0:00:00


## Step 3: Upload your data files
Run the cell, click **Choose files**, and select all your `.jsonl` files at once
(the 5 open-model "fingerprinting" files, plus the 2 Gemini "training" files if you want the bonus run).

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving gemini3.1protraining.jsonl to gemini3.1protraining.jsonl
Saving gemini3.8flashtraining.jsonl to gemini3.8flashtraining.jsonl
Saving gemma34bfingerprinting.jsonl to gemma34bfingerprinting.jsonl
Saving llama3.18bfingerprinting.jsonl to llama3.18bfingerprinting.jsonl
Saving llama3.23bfingerprinting.jsonl.json to llama3.23bfingerprinting.jsonl.json
Saving qwen3.8bfingerprinting.jsonl.json to qwen3.8bfingerprinting.jsonl.json
Saving qwen3.59bfingerprinting.jsonl to qwen3.59bfingerprinting.jsonl
['gemini3.1protraining.jsonl', 'gemini3.8flashtraining.jsonl', 'gemma34bfingerprinting.jsonl', 'llama3.18bfingerprinting.jsonl', 'llama3.23bfingerprinting.jsonl.json', 'qwen3.8bfingerprinting.jsonl.json', 'qwen3.59bfingerprinting.jsonl']


## Step 4: Choose which run you are doing and prepare the data
- `"open_models"`: 5 classes (gemma3-4b, llama3.1-8b, llama3.2-3b, qwen3-8b, qwen3.5-9b). **Do this one first.**
- `"gemini"`: 2 classes (gemini-3.1-pro vs gemini-3.8-flash). Bonus run.

**Never mix the two groups in one run.** They used different questions, so the model could "cheat" by recognizing the question instead of the writing style.

In [ ]:
import json, glob, random, collections, re

RUN = "open_models"          # change to "gemini" for the bonus run

if RUN == "open_models":
    FILE_PATTERN = "*fingerprinting*"
    MAX_TRAIN_PER_CLASS = 200     # keeps the classes roughly balanced
else:
    FILE_PATTERN = "*training*"
    MAX_TRAIN_PER_CLASS = 400

MAX_TOKENS = 384      # only the first 384 tokens of each answer are used
SEED = 42

# 1) load every record
rows = []
for f in sorted(glob.glob(FILE_PATTERN)):
    with open(f) as fh:
        for line in fh:
            if line.strip():
                rows.append(json.loads(line))
print("records loaded:", len(rows))

# 2) keep only records that have an actual answer (some Qwen3.5 answers are empty
#    because the model used all its tokens "thinking")
def has_answer(r):
    return (r.get("response_text") or "").strip() != "" and not r.get("error")
usable = [r for r in rows if has_answer(r)]
print("usable records:", len(usable), "| dropped:", len(rows) - len(usable))
print("usable per model:", dict(collections.Counter(r["model_key"] for r in usable)))

# 3) split by QUESTION, not by record.
#    Every question was asked many times, so a random split would put near-copies
#    of the same answer in both train and test, which makes the score look better than it is.
prompt_group = {r["prompt_id"]: r["prompt_group"] for r in rows}
rng = random.Random(SEED)
test_prompts = set()
for g in sorted(set(prompt_group.values())):
    ids = sorted(i for i, x in prompt_group.items() if x == g)
    rng.shuffle(ids)
    test_prompts |= set(ids[: max(1, round(len(ids) * 0.2))])

train_all = [r for r in usable if r["prompt_id"] not in test_prompts]
test = [r for r in usable if r["prompt_id"] in test_prompts]

# 4) cap each class in the training set so no model dominates
rng.shuffle(train_all)
count = collections.Counter()
train = []
for r in train_all:
    if count[r["model_key"]] < MAX_TRAIN_PER_CLASS:
        count[r["model_key"]] += 1
        train.append(r)

labels = sorted(set(r["model_key"] for r in usable))
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
print("\nclasses:", labels)
print("train examples:", len(train), dict(collections.Counter(r["model_key"] for r in train)))
print("test examples:", len(test), dict(collections.Counter(r["model_key"] for r in test)))

def family(name):
    return re.match(r"[a-z]+", name).group(0)   # gemma3-4b -> gemma, llama3.1-8b -> llama ...

records loaded: 3840
usable records: 3087 | dropped: 753
usable per model: {'gemma3-4b': 744, 'llama3.1-8b': 744, 'llama3.2-3b': 744, 'qwen3.5-9b': 137, 'qwen3-8b': 718}

classes: ['gemma3-4b', 'llama3.1-8b', 'llama3.2-3b', 'qwen3-8b', 'qwen3.5-9b']
train examples: 898 {'gemma3-4b': 200, 'llama3.2-3b': 200, 'llama3.1-8b': 200, 'qwen3-8b': 200, 'qwen3.5-9b': 98}
test examples: 697 {'gemma3-4b': 168, 'llama3.1-8b': 168, 'llama3.2-3b': 168, 'qwen3.5-9b': 39, 'qwen3-8b': 154}


## Step 5: A simple baseline to beat
This is a normal (non-LoRA) method that counts words. If LoRA can't beat this, that is still a valid result to report, and a good talking point for the mentor.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
Xtr = vec.fit_transform([r["response_text"] for r in train])
clf = LogisticRegression(max_iter=2000).fit(Xtr, [r["model_key"] for r in train])
base_pred = clf.predict(vec.transform([r["response_text"] for r in test]))
base_true = [r["model_key"] for r in test]
print("Baseline accuracy (exact model):", round(accuracy_score(base_true, base_pred), 3))
print("Baseline accuracy (family only):", round(accuracy_score([family(x) for x in base_true], [family(x) for x in base_pred]), 3))
print("Random guessing would be about", round(1 / len(labels), 2))

Baseline accuracy (exact model): 0.631
Baseline accuracy (family only): 0.819
Random guessing would be about 0.2


## Step 6: Load the small open model and attach LoRA
`Qwen/Qwen2.5-1.5B` is a small open-weight model (about 1.5 billion numbers). The LoRA "sticky notes" are the only part that gets trained.
**If you get a "CUDA out of memory" error later**, change the name below to `Qwen/Qwen2.5-0.5B` and re-run from this cell.

In [ ]:
!pip uninstall -y -q torchao # added this cuz the LoRA library doesnt work with the torchao library. so restart session under runtime and rerun steps 4-6

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B" # change from 1.5 to 0.5

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
tok.padding_side = "right"

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(labels), id2label=id2label, label2id=label2id
)
model = model.float()                       # full precision weights, training uses fp16 math on top
model.config.pad_token_id = tok.pad_token_id

lora = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                 # size of the sticky notes
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()          # you should see well under 5% being trained

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 8,802,688 || all params: 502,839,936 || trainable%: 1.7506


## Step 7: Turn the text into numbers the model can read

In [ ]:
from torch.utils.data import Dataset
from transformers import DataCollatorWithPadding

class AnswerDataset(Dataset):
    def __init__(self, records):
        self.items = []
        for r in records:
            enc = tok(r["response_text"].strip(), truncation=True, max_length=MAX_TOKENS)
            enc["labels"] = label2id[r["model_key"]]
            self.items.append(enc)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        return self.items[i]

train_ds = AnswerDataset(train)
test_ds = AnswerDataset(test)
collator = DataCollatorWithPadding(tokenizer=tok)
print(len(train_ds), "train /", len(test_ds), "test")

898 train / 697 test


## Step 8: Train
This is the actual fine-tuning. Expect roughly 15 to 40 minutes on a free T4.
The `loss` number printed as it goes should trend **down** (it's how wrong the model is).

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir="lora_out",
    per_device_train_batch_size=2, # change from 4 to 2
    gradient_accumulation_steps=8, # change from 4 to 8
    num_train_epochs=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, data_collator=collator)
trainer.train()

Step,Training Loss
10,50.684784
20,30.881656
30,17.823460
40,13.148355
50,11.114196
60,8.033372
70,4.810444
80,3.348901
90,3.313824
100,3.737859


TrainOutput(global_step=228, training_loss=7.257401894059098, metrics={'train_runtime': 430.5506, 'train_samples_per_second': 8.343, 'train_steps_per_second': 0.53, 'total_flos': 2591872919040000.0, 'train_loss': 7.257401894059098, 'epoch': 4.0})

## Step 9: Test on the questions the model has never seen

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(test_ds)
logits = out.predictions[0] if isinstance(out.predictions, tuple) else out.predictions
pred_ids = np.argmax(logits, axis=-1)
true_ids = [label2id[r["model_key"]] for r in test]

pred = [id2label[i] for i in pred_ids]
true = [id2label[i] for i in true_ids]

print("LoRA accuracy (exact model):", round(accuracy_score(true, pred), 3))
print("LoRA accuracy (family only):", round(accuracy_score([family(x) for x in true], [family(x) for x in pred]), 3))
print("Baseline was:", round(accuracy_score(base_true, base_pred), 3), "| random guessing:", round(1 / len(labels), 2))
print()
print(classification_report(true, pred, labels=labels, zero_division=0))
print("Confusion matrix (rows = real model, columns = what LoRA guessed):")
print("order:", labels)
print(confusion_matrix(true, pred, labels=labels))

LoRA accuracy (exact model): 0.673
LoRA accuracy (family only): 0.897
Baseline was: 0.631 | random guessing: 0.2

              precision    recall  f1-score   support

   gemma3-4b       0.97      0.87      0.92       168
 llama3.1-8b       0.46      0.56      0.51       168
 llama3.2-3b       0.51      0.38      0.44       168
    qwen3-8b       0.80      0.90      0.85       154
  qwen3.5-9b       0.60      0.69      0.64        39

    accuracy                           0.67       697
   macro avg       0.67      0.68      0.67       697
weighted avg       0.68      0.67      0.67       697

Confusion matrix (rows = real model, columns = what LoRA guessed):
order: ['gemma3-4b', 'llama3.1-8b', 'llama3.2-3b', 'qwen3-8b', 'qwen3.5-9b']
[[146   5   1  10   6]
 [  3  94  56  12   3]
 [  1  91  64  10   2]
 [  1   8   0 138   7]
 [  0   6   4   2  27]]


## Step 10: Try it on any text, and save your work

In [ ]:
def which_model(text):
    enc = tok(text, truncation=True, max_length=MAX_TOKENS, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits.float(), dim=-1)[0]
    ranked = sorted(zip(labels, probs.tolist()), key=lambda x: -x[1])
    return [(l, round(p, 3)) for l, p in ranked]

model.eval()
print(which_model(test[0]["response_text"]), "<- real answer:", test[0]["model_key"])

# save the LoRA "sticky notes" (small file) and download it
model.save_pretrained("my_lora_adapter")
!zip -r -q my_lora_adapter.zip my_lora_adapter
from google.colab import files
files.download("my_lora_adapter.zip")

[('gemma3-4b', 1.0), ('llama3.1-8b', 0.0), ('qwen3.5-9b', 0.0), ('qwen3-8b', 0.0), ('llama3.2-3b', 0.0)] <- real answer: gemma3-4b


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>